<a href="https://colab.research.google.com/github/Sandutta2020/Pyspark/blob/Master/Pyspark_in_google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

yothi.kommajosyula@ericsson.com

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"


import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark


In [ ]:
import requests
population_path = "https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-population.csv"
req = requests.get(population_path)
url_content = req.content
csv_file_name = 'state-population.csv'
csv_file = open(csv_file_name, 'wb')
csv_file.write(url_content)
csv_file.close()
df_population = spark.read.csv('/content/'+csv_file_name, header=True, inferSchema=True)

abbrevs_path = "https://raw.githubusercontent.com/jakevdp/data-USstates/master/state-abbrevs.csv"
req = requests.get(abbrevs_path)
url_content = req.content
csv_file_name = 'state-abbrevs.csv'
csv_file = open(csv_file_name, 'wb')
csv_file.write(url_content)
csv_file.close()
df_abbrevs = spark.read.csv('/content/'+csv_file_name, header=True, inferSchema=True)

In [ ]:
df_population.columns

['state/region', 'ages', 'year', 'population']

In [ ]:
#renaming the columns
df=df_population.withColumnRenamed('state/region','state_region')


In [ ]:
df_abbrevs.head(5)

[Row(state='Alabama', abbreviation='AL'),
 Row(state='Alaska', abbreviation='AK'),
 Row(state='Arizona', abbreviation='AZ'),
 Row(state='Arkansas', abbreviation='AR'),
 Row(state='California', abbreviation='CA')]

In [ ]:
df.createOrReplaceTempView("population1")

what is the 3rd highest state in terms of total population across all years

In [ ]:
spark.sql("select * from population1")

state_region,ages,year,population
AL,under18,2012,1117489.0
AL,total,2012,4817528.0
AL,under18,2010,1130966.0
AL,total,2010,4785570.0
AL,under18,2011,1125763.0
AL,total,2011,4801627.0
AL,total,2009,4757938.0
AL,under18,2009,1134192.0
AL,under18,2013,1111481.0
AL,total,2013,4833722.0


In [ ]:
# group by state_region
spark.sql('select sum(population) sp,state_region from population1 group by state_region')

sp,state_region
1.60203703E8,AZ
1.23345746E8,SC
1.34955449E8,LA
1.49209194E8,MN
2.50898559E8,NJ
1.6783359E7,DC
1.03052174E8,OR
2.15719854E8,VA
3.0673971E7,RI
1.21747078E8,KY


In [ ]:
# assigning rank
spark.sql('select row_number() over (order by sp desc ) rn ,sp, state_region from (select sum(population) sp,state_region from population1 group by state_region)')

rn,sp,state_region
1,NaN,PR
2,8.555469845E9,USA
3,1.04203578E9,CA
4,6.63243642E8,TX
5,5.62215404E8,NY
6,4.83990138E8,FL
7,3.72369723E8,IL
8,3.65556741E8,PA
9,3.39805369E8,OH
10,2.95570018E8,MI


In [ ]:
#now PR is showing 'NAN'  -- Need to check that
spark.sql("select sum(population) sp,state_region from population1 where state_region ='PR' group by state_region")


sp,state_region
6.6289715E7,PR


In [ ]:
spark.sql("select  NVL(population,0) from population1 where state_region ='PR'")

"nvl(population, 0)"
NaN
NaN
NaN
NaN
NaN
NaN
NaN
NaN
NaN
NaN


In [ ]:
# filling NAN with 0
df=df.fillna(value=0,subset=["population"])
df.createOrReplaceTempView("population1")

In [ ]:
spark.sql('select row_number() over (order by sp desc ) rn ,sp, state_region from (select sum(population) sp,state_region from population1 group by state_region)')

rn,sp,state_region
1,8.555469845E9,USA
2,1.04203578E9,CA
3,6.63243642E8,TX
4,5.62215404E8,NY
5,4.83990138E8,FL
6,3.72369723E8,IL
7,3.65556741E8,PA
8,3.39805369E8,OH
9,2.95570018E8,MI
10,2.53700202E8,GA


In [ ]:
# For 3rd highest
spark.sql('select state_region,sp from (select row_number() over (order by sp desc ) rn ,sp, state_region from (select sum(population) sp,state_region from population1 group by state_region)) where rn ==3')

state_region,sp
TX,6.63243642E8


Given a string s containing just the characters '(', ')', '{', '}', '[' and ']', determine if the input string is valid.

An input string is valid if:

Open brackets must be closed by the same type of brackets.
Open brackets must be closed in the correct order.
Every close bracket has a corresponding open bracket of the same type.

In [ ]:
str_input =list('[][())]{}()')
Open_stack=[]
for i in str_input:
    if i=='[' or i=='{' or i=='(':
        Open_stack.append(i)
    elif i==']' or i=='}' or i==')':
        if len(Open_stack)==0:
            print('not valid(bracket_mismatch)')
            break
        Open_bracket =Open_stack.pop()
        if (i==']' and Open_bracket=='[') or (i==')' and Open_bracket=='(') or (i=='}' and Open_bracket=='{'):
            continue
        else:
            print('not valid')
            break
    print(Open_stack)
if len(Open_stack)==0:
    print('valid')






['[']
['[']
['[', '(']
not valid
valid


In [ ]:
def check_valid_bracket(str_input):
    Open_stack=[]
    for i in str_input:
        if i=='[' or i=='{' or i=='(':
            Open_stack.append(i)
        elif i==']' or i=='}' or i==')':
            if len(Open_stack)==0:
                return 'not valid(started with end bracket)'
                break
            Open_bracket =Open_stack.pop()
            if (i==']' and Open_bracket=='[') or (i==')' and Open_bracket=='(') or (i=='}' and Open_bracket=='{'):
                continue
            else:
                return 'not valid'
                break
    if len(Open_stack)==0:
        return 'valid'

In [ ]:
print(check_valid_bracket('[][]{}()'))

valid


In [ ]:
print(check_valid_bracket('[][()]{}()'))

valid


In [ ]:
print(check_valid_bracket('[][())]{}()'))

not valid


In [ ]:
print(check_valid_bracket('][][())]{}()'))

not valid(started with end bracket)
